In [ ]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Path

skyline_path = 'data/MARTHA/DE17501_Martha_results_dotp.csv'
## Read data
skyline_data = pd.read_csv(skyline_path)
# ## For skyline data, remove column Precursor and drop duplicates
# skyline_data = skyline_data.drop(columns=['Precursor'])



# SDRF 
sdrf_data = pd.read_csvv
# Save the combined sdrf data to a file
# sdrf_data.to_csv('export/sdrf_MARTHA_combined.tsv', sep='\t', index=False)




In [ ]:
skyline_data[['Protein Name']].to_csv('export/protein_names.csv', index=False)
irt_tag_protein = skyline_data[skyline_data['Protein Name'].str.contains('iRT_Tag')] 
iRT_peptides = skyline_data['Peptide Sequence'][skyline_data['Protein Name'].str.contains('iRT_Tag')]
iRT_peptides = iRT_peptides.drop_duplicates().tolist()

print(iRT_peptides)

In [ ]:
# Plot distrion of Library Dot Product

# Plot distribution of Library Dot Product
plt.figure(figsize=(10, 6))
sns.histplot(skyline_data['Library Dot Product'], bins=20, kde=True)
plt.title('Distribution of Library Dot Product')
plt.xlabel('Library Dot Product')

In [ ]:
skyline_data

In [ ]:
skyline_data.tail(3000)

In [ ]:
import re

# Add column Isotope Label Type
skyline_data['Isotope Label Type'] = skyline_data['Precursor'].apply(lambda x: 'heavy' if re.search(r'heavy', str(x), re.IGNORECASE) else 'light')

# Filter out library dot product that is less than 0.6
skyline_data = skyline_data[skyline_data['Library Dot Product'] > 0.6]

# # Remove Normalied Area that is Nan
skyline_data = skyline_data[skyline_data['Normalized Area'].notna()]
# # Prepare pivot table, keeping all other column names as index except 'Isotope Label Type' and 'Intensity'
index_cols = [col for col in skyline_data.columns if col not in ['Isotope Label Type', 'Intensity']]
skyline_pivot = skyline_data.pivot_table(
    index=['Replicate', 'Peptide'],
    columns='Isotope Label Type',
    values='Normalized Area',
    aggfunc='first'
).reset_index()

# # Calculate lh_ratio
# skyline_pivot['lh_ratio'] = skyline_pivot['heavy'] / skyline_pivot['light']


skyline_pivot.head(20)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = skyline_pivot.groupby('Peptide').agg(
    heavy_count=pd.NamedAgg(column='heavy', aggfunc=lambda x: x.notna().sum()),
    light_count=pd.NamedAgg(column='light', aggfunc=lambda x: x.notna().sum())
).reset_index()

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
plt.scatter(peptide_counts['light_count'], peptide_counts['heavy_count'], alpha=0.5)
plt.axvline(700, color='g', linestyle='--', label='Light count = 700')
plt.axhline(700, color='g', linestyle='--', label='Heavy count = 700')
plt.xlabel('Light Count')
plt.ylabel('Heavy Count')
plt.title('Peptide Detection Counts: Light vs Heavy')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Sumamrise Protein and Peptide that has been detected the least
# Count the number of unique peptides and proteins in the data
num_unique_peptides = skyline_data['Peptide'].nunique()

# If Protein Name is not present in peptide_counts (from pivot), map it from skyline_data
if 'Protein Name' in skyline_data.columns:
    num_unique_proteins = skyline_data['Protein Name'].nunique()
else:
    num_unique_proteins = 0

print(f"Number of unique peptides: {num_unique_peptides}")
print(f"Number of unique proteins: {num_unique_proteins}")

# For peptide_counts, Protein Name is missing, so we need to map peptide->protein using skyline_data
peptide_to_protein = skyline_data[['Peptide', 'Protein Name']].drop_duplicates().set_index('Peptide')['Protein Name']

# Add Protein Name to peptide_counts for peptide-based summaries
peptide_counts = peptide_counts.merge(peptide_to_protein, left_on='Peptide', right_index=True, how='left')

# Peptides detected only once in heavy (rare peptides)
least_detected_peptides = peptide_counts[peptide_counts['heavy_count'] == 1]['Peptide'].nunique()
least_detected_proteins = peptide_counts[peptide_counts['heavy_count'] == 1]['Protein Name'].nunique()

# Counting unique peptides with only 1 detection in light/heavy signal
least_detected_light = peptide_counts[peptide_counts['light_count'] == 1].shape[0]
least_detected_heavy = peptide_counts[peptide_counts['heavy_count'] == 1].shape[0]

print(f"Number of peptides detected only once in heavy: {least_detected_peptides}")
print(f"Number of proteins (containing such peptides) detected only once in heavy: {least_detected_proteins}")

print(f"Number of peptides detected only once in light: {least_detected_light}")
print(f"Number of peptides detected only once in heavy: {least_detected_heavy}")




In [ ]:
# Set the cut off at 700 on both light and heavy count and then summarise the peptide list and plot the scatter again
peptide_counts_cutoff = peptide_counts[(peptide_counts['heavy_count'] > 700) & (peptide_counts['light_count'] > 700)]



# Summarise how many peptide and protein are detected in the cutoff
peptide_counts_cutoff['Peptide'].nunique()
print(f"Number of peptides detected in the cutoff: {peptide_counts_cutoff['Peptide'].nunique()}")

# Fitlered peptides list 
selected_peptides = peptide_counts_cutoff['Peptide'].drop_duplicates().tolist()
peptide_counts_cutoff['Protein Name'].nunique()
print(f"Number of proteins detected in the cutoff: {peptide_counts_cutoff['Protein Name'].nunique()}")

# Plot the scatter again
plt.figure(figsize=(10, 6))
sns.scatterplot(x='heavy_count', y='light_count', data=peptide_counts_cutoff)
plt.title('Scatter plot of Heavy Count vs Light Count per Peptide cutoff dotp = 0.6')
plt.xlabel('heavy_count')
plt.ylabel('light_count')
plt.show()

In [ ]:
skyline_pivot.head(200)

In [ ]:
sdrf_data.head(200)

In [ ]:
# Filtered skyline_data with selected_peptides
skyline_data_filtered = skyline_data[skyline_data['Peptide'].isin(selected_peptides)]

# Merge with SDRF
skyline_data_filtered = skyline_data_filtered.merge(sdrf_data, left_on='Replicate', right_on='Run', how='left')

# Plot RatioLighttoHeavy with boxplot for each Replicate
plt.figure(figsize=(16, 6))
sns.boxplot(
    data=skyline_data_filtered,
    x='Replicate',
    y='RatioLightToHeavy'
)
plt.title('Boxplot of RatioLighttoHeavy for Each Replicate')
plt.xlabel('Replicate')
plt.ylabel('RatioLighttoHeavy')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# 